In [ ]:
# 📦 Import Required Libraries
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 📁 Load Dataset
df = pd.read_csv("EVData.csv")

# 📊 Initial Data Overview
print("Top 5 rows of the dataset:")
print(df.head())
print("\nShape of the dataset:", df.shape)
print("\nInfo:")
df.info()

# 🧼 Check Missing Values
print("\nMissing values per column:")
print(df.isnull().sum())

# ⚠️ Identify Outliers in 'Percent Electric Vehicles'
Q1 = df['Percent Electric Vehicles'].quantile(0.25)
Q3 = df['Percent Electric Vehicles'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("\nOutlier bounds:")
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

outliers = df[(df['Percent Electric Vehicles'] < lower_bound) | (df['Percent Electric Vehicles'] > upper_bound)]
print("Number of outliers:", outliers.shape[0])

# 🧹 Data Cleaning

# Convert 'Date' to datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df[df['Date'].notnull()]  # Drop rows with invalid dates

# Drop rows with missing EV Total (target)
df = df[df['Electric Vehicle (EV) Total'].notnull()]

# Fill missing 'County' and 'State'
df['County'] = df['County'].fillna('Unknown')
df['State'] = df['State'].fillna('Unknown')

print("\nMissing values after filling:")
print(df[['County', 'State']].isnull().sum())

# ✂️ Remove Outliers by Capping
df['Percent Electric Vehicles'] = np.where(
    df['Percent Electric Vehicles'] > upper_bound, upper_bound,
    np.where(df['Percent Electric Vehicles'] < lower_bound, lower_bound, df['Percent Electric Vehicles'])
)

# ✅ Verify No Outliers Remain
outliers_capped = df[(df['Percent Electric Vehicles'] < lower_bound) | (df['Percent Electric Vehicles'] > upper_bound)]
print("Number of outliers after capping:", outliers_capped.shape[0])


Top 5 rows of the dataset:
                Date          County State Vehicle Primary Use  \
0  September 30 2022       Riverside    CA           Passenger   
1   December 31 2022  Prince William    VA           Passenger   
2    January 31 2020          Dakota    MN           Passenger   
3       June 30 2022           Ferry    WA               Truck   
4       July 31 2021         Douglas    CO           Passenger   

  Battery Electric Vehicles (BEVs) Plug-In Hybrid Electric Vehicles (PHEVs)  \
0                                7                                        0   
1                                1                                        2   
2                                0                                        1   
3                                0                                        0   
4                                0                                        1   

  Electric Vehicle (EV) Total Non-Electric Vehicle Total Total Vehicles  \
0                         